# z623 - Ensamble final: las 3 mejores versiones
Promedio de los 3 mejores resultados confirmados: Regresion Lineal (`LR01`, 0.231), AutoGluon (`AutoGluon-01`, 0.251) y LightGBM (`LGB07_WF`, 0.251).

In [1]:
import os
os.makedirs('/home/ds/exp/AutoGluon-01', exist_ok=True)

In [2]:
import os
import pandas as pd

In [3]:
PARAM = {
    'experimento': 'ENS03_MEJORES3',
    'kaggle_competition': 'labo-iii-2026-ba',
    'submits_a_promediar': {
        'regresion_lineal': '/home/ds/exp/LR01/linreg.csv',
        'autogluon': '/home/ds/exp/AutoGluon-01/AutoGluon_RMSE.csv',
        'lightgbm': '/home/ds/exp/LGB07_WF/LGB07_WF_submit.csv',
    }
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/ENS03_MEJORES3


## 1. Verificar que los 3 archivos existan

In [4]:
for nombre, p in PARAM['submits_a_promediar'].items():
    print(nombre, p, os.path.isfile(p))

regresion_lineal /home/ds/exp/LR01/linreg.csv True
autogluon /home/ds/exp/AutoGluon-01/AutoGluon_RMSE.csv True
lightgbm /home/ds/exp/LGB07_WF/LGB07_WF_submit.csv True


## 2. Cargar y promediar (simple, mismo peso para los 3)

In [5]:
tablas = {nombre: pd.read_csv(p) for nombre, p in PARAM['submits_a_promediar'].items()}

base = list(tablas.values())[0][["product_id"]].copy()
for nombre, t in tablas.items():
    base = base.merge(t.rename(columns={"tn": f"tn_{nombre}"}), on="product_id", how="left")

cols_tn = [c for c in base.columns if c.startswith("tn_")]
base["tn"] = base[cols_tn].mean(axis=1)

submit = base[["product_id", "tn"]]
print(submit.shape)
submit.head()

(780, 2)


,product_id,tn
0,20001,1181.918116
1,20002,1092.362131
2,20003,762.430805
3,20004,572.825830
4,20005,526.020417


## 3. Guardar y submit

In [6]:
archivo_submit = os.path.join(ruta, f"{PARAM['experimento']}_submit.csv")
submit.to_csv(archivo_submit, index=False)
print(archivo_submit)

/home/ds/exp/ENS03_MEJORES3/ENS03_MEJORES3_submit.csv


In [7]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

kaggle_submit(PARAM['kaggle_competition'], archivo_submit, f"{PARAM['experimento']} ensamble LR01+AutoGluon+LGB07_WF")

100%|██████████| 18.7k/18.7k [00:00<00:00, 57.1kB/s]


94 submissions remaining today.
Successfully submitted to Labo III, 2026 BA